# Step 3: Cell-type proportions and spatial neighborhood enrichment

Starting from the labeled AnnData produced by Step 2, build a spatial graph
per sample and quantify cell-type co-localization with a permutation-based
neighborhood-enrichment test (`squidpy.gr.nhood_enrichment`). Per-pair
log2(observed / expected) values are summarized across samples to obtain a
consensus map, and DX -> PT changes are compared between patients with
Mann-Whitney U tests.

CSV outputs in `data/processed/`:

- `ext_fig2_distances.csv` (per-sample log2(O/E) for all cell-type pairs)
- `fig3b_consensus.csv` (consensus log2(O/E) across samples)
- `fig3c_delta_PatientN.csv` (DX -> PT delta for each of Patient1..5)
- `fig3c_mannwhitney_stats.csv` (Mann-Whitney tests across patient groups)

Figure outputs include per-sample spatial maps colored by cell type, a
consensus 3D bar plot of log2(O/E), and per-sample 3D bar plots in which
height encodes |log2(O/E)| and color encodes the sign.

## Setup and imports

In [ ]:
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.cm as mcm
import matplotlib.colors as mcolors
import seaborn as sns
import pandas as pd
import numpy as np
import os
import math
import anndata as ad
from scipy.stats import wilcoxon, mannwhitneyu
from scipy import sparse
from statsmodels.stats.multitest import multipletests, fdrcorrection
from joblib import Parallel, delayed
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
from matplotlib.patches import Circle
import plotly.graph_objects as go

In [ ]:
sc.settings.verbosity = 3
np.random.seed(26)

# Path config
indir   = '/path/to/integrated/processed_data/'
csvdir  = '../data/processed/'
figdir  = '../figures/'
os.makedirs(csvdir, exist_ok=True)
os.makedirs(figdir, exist_ok=True)

plt.rcParams['savefig.transparent'] = True
plt.rcParams['savefig.dpi'] = 600
mpl.rcParams['font.family'] = 'Helvetica'
mpl.rcParams['font.size'] = 6
sc.settings.figdir = figdir

# Patient color palette (used in optional per-patient overlays)
patient_colors = {
    'Patient1': '#d9d9d9', 'Patient4': '#bdbdbd', 'Patient5': '#969696',
    'Patient2': '#636363', 'Patient3': '#252525',
}

FONT_SIZE = 5
FIGSIZE_IN = (7 / 2.54, 7 / 2.54)
celltype_order = ['Neuroblast', 'B', 'T', 'Schwann', 'Endothelial', 'Fibroblast', 'Macrophage']

## Load labeled h5ad

In [ ]:
adata = sc.read_h5ad(indir + 'xenium_integrated_labeled.h5ad')

## Per-sample spatial maps colored by cell type

Renders one PNG per sample from the per-cell spatial coordinates.

In [ ]:
# Per-sample spatial scatter plot coloured by main cell type.
sample_key, patient_key, timepoint_key, group_key = 'sample', 'patient', 'timepoint', 'celltype'
scalebar_len = 500

if 'celltype_colors' in adata.uns and pd.api.types.is_categorical_dtype(adata.obs[group_key]):
    cats   = adata.obs[group_key].cat.categories
    colors = adata.uns['celltype_colors']
    color_map = {c: colors[i] for i, c in enumerate(cats)}
else:
    color_map = {c: plt.cm.tab20(i % 20) for i, c in enumerate(sorted(pd.unique(adata.obs[group_key])))}

meta = adata.obs[[sample_key, patient_key, timepoint_key]].drop_duplicates()
mins, maxs = [], []
for s in meta[sample_key].unique():
    coords = adata[adata.obs[sample_key] == s].obsm['spatial']
    mins.append(coords.min(axis=0)); maxs.append(coords.max(axis=0))
global_min = np.vstack(mins).min(axis=0); global_max = np.vstack(maxs).max(axis=0)

for _, row in meta.iterrows():
    sample, patient, tp = row[sample_key], row[patient_key], row[timepoint_key]
    sub = adata[adata.obs[sample_key] == sample]
    coords = sub.obsm['spatial']
    labels = sub.obs[group_key].astype(str).values
    cell_colors = [color_map.get(x, 'gray') for x in labels]
    fig, ax = plt.subplots(figsize=(9, 9))
    ax.scatter(coords[:, 0], coords[:, 1], s=2, c=cell_colors, linewidths=0, alpha=0.8)
    ax.set_xlim(global_min[0], global_max[0]); ax.set_ylim(global_min[1], global_max[1])
    ax.set_aspect('equal', adjustable='box')
    ax.set_title(f'{patient} {tp}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if str(tp).upper() == 'DX':
        x0 = global_min[0] + 0.05 * (global_max[0] - global_min[0])
        y0 = global_max[1] - 0.05 * (global_max[1] - global_min[1])
        ax.plot([x0, x0 + scalebar_len], [y0, y0], color='black', linewidth=2)
        ax.text(x0, y0 - 0.02 * (global_max[1] - global_min[1]),
                f'{scalebar_len} µm', fontsize=8, ha='left', va='top')
    plt.tight_layout()
    plt.savefig(figdir + f'spatial_{patient}_{tp}.png',
                format='png', bbox_inches='tight', transparent=True, dpi=600)
    plt.show(); plt.close(fig)

## Neighborhood enrichment helpers

Permutation-based neighborhood enrichment with label shuffling on a fixed
spatial graph; returns a tidy `df_pairs` table per sample.

In [ ]:
def _apply_heatmap_tick_style(ax):
    ax.tick_params(axis='both', which='both', width=0.5, length=2)

def _to_dense(x):
    return x.A if sparse.issparse(x) else np.asarray(x)

def _stars_from_q(q):
    if pd.isna(q): return ''
    if q < 1e-3: return '***'
    if q < 1e-2: return '**'
    if q < 5e-2: return '*'
    return ''

def _stars_from_pct(pct):
    if pd.isna(pct): return ''
    if pct >= 75: return '***'
    if pct >= 50: return '**'
    if pct >= 30: return '*'
    return ''

def sign_consistency(x):
    return (x > 0).all() or (x < 0).all()

def get_common_categories(obs_series):
    if pd.api.types.is_categorical_dtype(obs_series):
        return list(obs_series.cat.categories)
    return sorted(pd.unique(obs_series))

def build_subtype_order_from_parent(adata, parent_order, subtype_key='celltypes_all', parent_key='celltype'):
    df_map = adata.obs[[subtype_key, parent_key]].dropna().drop_duplicates()
    out = []
    for parent in parent_order:
        subtypes = df_map.loc[df_map[parent_key] == parent, subtype_key].unique().tolist()
        out.extend(sorted(subtypes))
    return out

def _interaction_count_matrix_from_adj(adj_csr, codes, n_cls):
    if not sparse.isspmatrix_csr(adj_csr):
        adj_csr = adj_csr.tocsr()
    indptr, indices = adj_csr.indptr, adj_csr.indices
    counts = np.zeros((n_cls, n_cls), dtype=np.float64)
    for i in range(adj_csr.shape[0]):
        src = codes[i]
        nbr_idx = indices[indptr[i]:indptr[i + 1]]
        if nbr_idx.size == 0:
            continue
        nbr_labels = codes[nbr_idx]
        counts[src, :] += np.bincount(nbr_labels, minlength=n_cls).astype(np.float64)
    return counts

In [ ]:
def compute_nhood_pairs(adata_sub, group_key='celltype', n_perms=2000, seed=0, alpha=0.05,
                         pseudocount=1.0, coord_type='generic', spatial_key='spatial',
                         neigh_kwargs=None, connectivities_key='spatial_connectivities', n_jobs=8):
    """Spatial graph + permutation neighborhood enrichment. Returns df_pairs tidy table."""
    if neigh_kwargs is None:
        neigh_kwargs = {}
    sq.gr.spatial_neighbors(adata_sub, coord_type=coord_type, spatial_key=spatial_key, **neigh_kwargs)
    sq.gr.nhood_enrichment(adata_sub, cluster_key=group_key, n_perms=n_perms, seed=seed, copy=False)

    key = f'{group_key}_nhood_enrichment'
    obs = _to_dense(adata_sub.uns[key]['count']).astype(float)
    cats = adata_sub.obs[group_key]
    if not pd.api.types.is_categorical_dtype(cats):
        cats = cats.astype('category')
        adata_sub.obs[group_key] = cats
    labels = list(cats.cat.categories)
    n_cls = len(labels)

    adj = adata_sub.obsp[connectivities_key]
    if not sparse.isspmatrix_csr(adj):
        adj = adj.tocsr()
    base_codes = cats.cat.codes.to_numpy()
    if np.any(base_codes < 0):
        raise ValueError(f'{group_key} contains NaN/unassigned categories.')

    seeds = seed + np.arange(n_perms)

    def _perm_chunk(seed_chunk):
        exp_local = np.zeros((n_cls, n_cls), dtype=np.float64)
        exp_sq_local = np.zeros((n_cls, n_cls), dtype=np.float64)
        ge_local = np.zeros((n_cls, n_cls), dtype=np.float64)
        le_local = np.zeros((n_cls, n_cls), dtype=np.float64)
        for s in seed_chunk:
            perm_codes = base_codes.copy()
            rng_local = np.random.default_rng(s)
            rng_local.shuffle(perm_codes)
            perm_counts = _interaction_count_matrix_from_adj(adj, perm_codes, n_cls)
            exp_local += perm_counts
            exp_sq_local += perm_counts ** 2
            ge_local += (perm_counts >= obs)
            le_local += (perm_counts <= obs)
        return exp_local, exp_sq_local, ge_local, le_local

    if n_jobs is None or n_jobs <= 1:
        exp_sum, exp_sq_sum, ge_sum, le_sum = _perm_chunk(seeds)
    else:
        n_jobs_eff = min(int(n_jobs), n_perms)
        parts = Parallel(n_jobs=n_jobs_eff, backend='loky')(
            delayed(_perm_chunk)(chunk) for chunk in np.array_split(seeds, n_jobs_eff)
        )
        exp_sum    = np.sum([p[0] for p in parts], axis=0)
        exp_sq_sum = np.sum([p[1] for p in parts], axis=0)
        ge_sum     = np.sum([p[2] for p in parts], axis=0)
        le_sum     = np.sum([p[3] for p in parts], axis=0)

    exp_perm = exp_sum / float(n_perms)
    exp_var  = np.clip((exp_sq_sum / float(n_perms)) - (exp_perm ** 2), 0, None)
    exp_sd   = np.sqrt(exp_var)
    z_perm   = (obs - exp_perm) / (exp_sd + 1e-9)
    log2_oe  = np.log2((obs + pseudocount) / (exp_perm + pseudocount))

    p_upper = (ge_sum + 1.0) / (n_perms + 1.0)
    p_lower = (le_sum + 1.0) / (n_perms + 1.0)
    p = np.clip(2.0 * np.minimum(p_upper, p_lower), 0, 1)

    reject_flat, q_flat = fdrcorrection(p.ravel(), alpha=alpha)
    significant = reject_flat.reshape(p.shape)
    q = q_flat.reshape(p.shape)

    df_pairs = pd.DataFrame({
        'source':   np.repeat(labels, len(labels)),
        'neighbor': labels * len(labels),
        'obs':      obs.ravel(),
        'exp_perm': exp_perm.ravel(),
        'log2_oe':  log2_oe.ravel(),
        'z':        z_perm.ravel(),
        'p':        p.ravel(),
        'q':        q.ravel(),
        'significant': significant.ravel(),
    })
    df_pairs['low_expected'] = df_pairs['exp_perm'] < 5
    return df_pairs

In [ ]:
def build_consensus_table(df_pairs, min_samples_sig=0.5, include_dispersion=True, out_csv=None):
    if df_pairs.empty:
        raise ValueError('df_pairs is empty.')
    x = df_pairs.copy()
    x['sig_valid'] = (x['significant'] & ~x['low_expected']).astype(float)
    agg = {
        'mean_log2_oe': ('log2_oe', 'mean'),
        'mean_obs':     ('obs', 'mean'),
        'pct_sig':      ('sig_valid', 'mean'),
    }
    if include_dispersion:
        agg.update({
            'sd_log2_oe':  ('log2_oe', 'std'),
            'iqr_log2_oe': ('log2_oe', lambda v: np.nanpercentile(v, 75) - np.nanpercentile(v, 25)),
        })
    consensus = x.groupby(['source', 'neighbor']).agg(**agg).reset_index()
    consensus['pct_sig'] = 100 * consensus['pct_sig']
    consensus['consensus_sig'] = consensus['pct_sig'] >= (min_samples_sig * 100)
    consensus['min_samples_sig_threshold_pct'] = min_samples_sig * 100
    if out_csv is not None:
        consensus.to_csv(out_csv, index=False)
    return consensus

def get_consistent_pairs(df_pairs, focus_labels=None):
    sig = df_pairs.query('significant and not low_expected').copy()
    if focus_labels is not None:
        focus = set(focus_labels)
        sig = sig[sig['source'].isin(focus) | sig['neighbor'].isin(focus)].copy()
    consistency = sig.groupby(['source','neighbor'])['log2_oe'].agg(
        n_sig='size',
        all_same_sign=sign_consistency,
        direction=lambda v: 'enriched' if (v > 0).all() else ('depleted' if (v < 0).all() else 'mixed'),
    ).reset_index()
    return consistency[consistency['all_same_sign']].copy()

## Per-sample neighborhood enrichment (Ext Fig 2)

Iterates over samples, computes per-sample `df_pairs`, and concatenates the
results to `ext_fig2_distances.csv`.

In [ ]:
all_df = []
samples = adata.obs['sample'].unique().tolist()
for sample in samples:
    print(f'Processing {sample}...')
    adata_sub = adata[adata.obs['sample'] == sample].copy()
    if adata_sub.n_obs < 100:
        continue
    df_pairs = compute_nhood_pairs(
        adata_sub, group_key='celltype', n_perms=5000, seed=0, alpha=0.05,
        pseudocount=1.0, n_jobs=8,
    )
    df_pairs['sample'] = sample
    all_df.append(df_pairs)

df_pairs_all = pd.concat(all_df, ignore_index=True) if all_df else pd.DataFrame()
df_pairs_all.to_csv(csvdir + 'ext_fig2_distances.csv', index=False)

## Per-patient DX -> PT deltas and Mann-Whitney tests (Fig 3c)

For each patient, take the DX -> PT difference in log2(O/E) for every
cell-type pair and save to `fig3c_delta_PatientN.csv`. Across patients, test
selected immune cell-type pairs with a Mann-Whitney U test contrasting
immune-rebuilding (Patients 1-3) and immune-desertification (Patients 4-5)
trajectories; results are written to `fig3c_mannwhitney_stats.csv`.

In [ ]:
df_pairs_all = pd.read_csv(csvdir + 'ext_fig2_distances.csv')

# Sample IDs are formatted as 'PatientN_DX' / 'PatientN_PT'.
df_pairs_meta = df_pairs_all.copy()
parts = df_pairs_meta['sample'].astype(str).str.rsplit('_', n=1, expand=True)
df_pairs_meta['patient']   = parts[0].str.strip()
df_pairs_meta['timepoint'] = parts[1].str.strip().str.upper()
df_pairs_meta = df_pairs_meta[df_pairs_meta['timepoint'].isin(['DX', 'PT'])].copy()

for patient in sorted(df_pairs_meta['patient'].dropna().unique()):
    df_p = df_pairs_meta[df_pairs_meta['patient'] == patient].copy()
    dx = df_p[df_p['timepoint'] == 'DX'].set_index(['source', 'neighbor'])
    pt = df_p[df_p['timepoint'] == 'PT'].set_index(['source', 'neighbor'])
    paired = pt[['log2_oe']].rename(columns={'log2_oe': 'pt'}).join(
        dx[['log2_oe']].rename(columns={'log2_oe': 'dx'}), how='inner')
    paired['delta'] = paired['pt'] - paired['dx']
    delta = paired['delta'].reset_index()
    delta = delta[['source', 'neighbor', 'delta']]
    delta.to_csv(csvdir + f'fig3c_delta_{patient}.csv', index=False)

In [ ]:
# Mann-Whitney tests of DX -> PT delta log2(O/E) for selected T / B /
# Macrophage interactions, comparing immune-rebuilding vs
# immune-desertification trajectories across patients.
df_pairs_all = pd.read_csv(csvdir + 'ext_fig2_distances.csv')
x = df_pairs_all.copy()
x['sample'] = x['sample'].astype(str).str.strip()
parts = x['sample'].str.rsplit('_', n=1, expand=True)
x['patient']   = parts[0].str.strip()
x['timepoint'] = parts[1].str.strip().str.upper()
x = x[x['timepoint'].isin(['DX', 'PT'])].copy()

targets = [
    ('B', 'B'), ('B', 'T'), ('T', 'B'), ('T', 'T'),
    ('Macrophage', 'T'), ('T', 'Macrophage'),
]
target_df = pd.DataFrame(targets, columns=['source', 'neighbor'])
x = x.merge(target_df, on=['source', 'neighbor'], how='inner')

wide = x.pivot_table(index=['patient','source','neighbor'], columns='timepoint',
                     values='log2_oe', aggfunc='mean').reset_index()
wide = wide.dropna(subset=['DX', 'PT']).copy()
wide['delta_PT_minus_DX'] = wide['PT'] - wide['DX']

rebuilding_patients = ['Patient1', 'Patient2', 'Patient3']
desert_patients     = ['Patient4', 'Patient5']
wide['trajectory_group'] = np.where(
    wide['patient'].isin(rebuilding_patients), 'immune_rebuilding',
    np.where(wide['patient'].isin(desert_patients), 'immune_desertification', np.nan))
wide = wide.dropna(subset=['trajectory_group']).copy()

rows = []
for s, n in targets:
    d  = wide[(wide['source'] == s) & (wide['neighbor'] == n)]
    rb = d.loc[d['trajectory_group'] == 'immune_rebuilding',     'delta_PT_minus_DX'].values
    ds = d.loc[d['trajectory_group'] == 'immune_desertification','delta_PT_minus_DX'].values
    if len(rb) == 0 or len(ds) == 0:
        rows.append({'source': s, 'neighbor': n, 'n_rebuilding': len(rb), 'n_desertification': len(ds),
                     'U_stat': np.nan, 'p_one_sided_less': np.nan, 'p_two_sided': np.nan,
                     'mean_delta_rebuilding': np.nan, 'mean_delta_desertification': np.nan})
        continue
    U_less, p_less = mannwhitneyu(rb, ds, alternative='less')
    _,       p_2s  = mannwhitneyu(rb, ds, alternative='two-sided')
    rows.append({
        'source': s, 'neighbor': n,
        'n_rebuilding': int(len(rb)), 'n_desertification': int(len(ds)),
        'mean_delta_rebuilding': float(np.mean(rb)), 'mean_delta_desertification': float(np.mean(ds)),
        'median_delta_rebuilding': float(np.median(rb)), 'median_delta_desertification': float(np.median(ds)),
        'U_stat': float(U_less), 'p_one_sided_less': float(p_less), 'p_two_sided': float(p_2s),
        'min_possible_one_sided_p_with_n3_vs_n2': 0.1,
    })

stats_df = pd.DataFrame(rows)
valid = stats_df['p_one_sided_less'].notna()
if valid.any():
    _, qvals, _, _ = multipletests(stats_df.loc[valid, 'p_one_sided_less'], method='fdr_bh')
    stats_df.loc[valid, 'p_one_sided_less_fdr_bh'] = qvals
stats_df.to_csv(csvdir + 'fig3c_mannwhitney_stats.csv', index=False)
display(stats_df)

## Consensus neighborhood enrichment across samples (Fig 3b)

Mean log2(O/E) per cell-type pair across samples, together with the fraction
of samples in which the pair was significantly enriched or depleted.

In [ ]:
df_pairs_all = pd.read_csv(csvdir + 'ext_fig2_distances.csv')
consensus = build_consensus_table(df_pairs_all, min_samples_sig=0.5, include_dispersion=True)
consensus.to_csv(csvdir + 'fig3b_consensus.csv', index=False)
display(consensus.head())

## 3D neighborhood-enrichment bar plots

For each cell-type pair, bar height encodes |log2(O/E)| and color encodes
the sign (red = enriched, blue = depleted). One figure is produced for the
consensus values across samples and one per individual sample.

In [ ]:
def val_to_color(val, norm, cmap=mcm.RdBu_r):
    rgba = cmap(norm(val))
    r, g, b = [int(c * 255) for c in rgba[:3]]
    return f'rgb({r},{g},{b})'

def add_full_bar(fig, xi, yi, h, color):
    x0, x1 = xi - 0.4, xi + 0.4
    y0, y1 = yi - 0.4, yi + 0.4
    vx = [x0, x1, x1, x0,  x0, x1, x1, x0]
    vy = [y0, y0, y1, y1,  y0, y0, y1, y1]
    vz = [0,  0,  0,  0,   h,  h,  h,  h]
    i_idx = [0, 0,  4, 4,  0, 0,  2, 2,  0, 0,  1, 1]
    j_idx = [1, 2,  5, 6,  1, 5,  3, 7,  3, 7,  2, 6]
    k_idx = [2, 3,  6, 7,  5, 4,  7, 6,  7, 4,  6, 5]
    fig.add_trace(go.Mesh3d(x=vx, y=vy, z=vz, i=i_idx, j=j_idx, k=k_idx,
                            color=color, opacity=1, flatshading=True,
                            lighting=dict(ambient=0.9, diffuse=0.1), showscale=False))

def add_bar_edges(fig, xi, yi, h, color='black', width=1):
    x0, x1 = xi - 0.4, xi + 0.4
    y0, y1 = yi - 0.4, yi + 0.4
    corners = [(x0,y0), (x1,y0), (x1,y1), (x0,y1)]
    ex, ey, ez = [], [], []
    for z_level in [0, h]:
        for i in range(4):
            j = (i + 1) % 4
            ex += [corners[i][0], corners[j][0], None]
            ey += [corners[i][1], corners[j][1], None]
            ez += [z_level, z_level, None]
    for cx, cy in corners:
        ex += [cx, cx, None]; ey += [cy, cy, None]; ez += [0, h, None]
    fig.add_trace(go.Scatter3d(x=ex, y=ey, z=ez, mode='lines',
                               line=dict(color=color, width=width),
                               showlegend=False, hoverinfo='none'))

def add_highlight_ring(fig, xi, yi, h, color='black', radius=0.38, width=4, n_pts=80):
    theta = np.linspace(0, 2 * np.pi, n_pts)
    fig.add_trace(go.Scatter3d(
        x=(xi + radius * np.cos(theta)).tolist(),
        y=(yi + radius * np.sin(theta)).tolist(),
        z=[h + 0.01] * n_pts, mode='lines',
        line=dict(color=color, width=width),
        showlegend=False, hoverinfo='none',
    ))

In [ ]:
# Consensus 3D bar plot: bar height = |log2(O/E)|; bar colour encodes the
# sign through a symmetric TwoSlopeNorm.
df = pd.read_csv(csvdir + 'fig3b_consensus.csv')
cats = [c for c in celltype_order if c in df['source'].unique()]
n = len(cats)
mat_log2 = (df.pivot(index='source', columns='neighbor', values='mean_log2_oe')
              .reindex(cats).reindex(cats, axis=1).fillna(0).values)
xpos, ypos = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
xpos = xpos.flatten(); ypos = ypos.flatten()
dz = mat_log2.flatten()

abs_max = float(np.nanmax(np.abs(dz))) if len(dz) else 1.0
norm = mcolors.TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
z_range = [0, float(np.ceil(abs_max * 2) / 2)]   # round to nearest 0.5

_elev, _azim, _r = 40, -215, 4
_eye = dict(x=_r*math.cos(math.radians(_elev))*math.cos(math.radians(_azim)),
            y=_r*math.cos(math.radians(_elev))*math.sin(math.radians(_azim)),
            z=_r*math.sin(math.radians(_elev)))
_font = dict(family='Helvetica', size=5, color='black')
axis_style = dict(showbackground=False, showgrid=False, zeroline=False,
                  showline=True, linecolor='black', tickcolor='black', tickfont=_font)

# Cell-type pairs highlighted on the 3D plot.
highlight_pairs = [
    ('B','T'), ('B','B'), ('T','T'), ('T','B'),
    ('B','Neuroblast'), ('Neuroblast','B'), ('T','Neuroblast'), ('Neuroblast','T'),
]

fig = go.Figure()
for i in range(len(dz)):
    if dz[i] == 0 or np.isnan(dz[i]):
        continue
    xi, yi = int(xpos[i]), int(ypos[i])
    val = float(dz[i]); h = float(np.abs(val))
    add_full_bar(fig, xi, yi, h, val_to_color(val, norm))
    add_bar_edges(fig, xi, yi, h)
for src, nbr in highlight_pairs:
    if src in cats and nbr in cats:
        xi, yi = cats.index(src), cats.index(nbr)
        h = float(np.abs(dz[xi * n + yi]))
        if h > 0:
            add_highlight_ring(fig, xi, yi, h, color='white', radius=0.2, width=6)

fig.update_layout(
    title='Consensus 3D heatmap (red = enriched, blue = depleted)',
    scene=dict(aspectmode='manual', aspectratio=dict(x=1.2, y=1.2, z=1.0),
               xaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats),
               yaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats),
               zaxis=dict(**axis_style, range=z_range,
                          title=dict(text='|log2(O/E)|', font=_font)),
               camera=dict(eye=_eye), bgcolor='white'),
    paper_bgcolor='white', plot_bgcolor='white', width=800, height=950,
)
fig.write_image(figdir + 'nhood_enrichment_3d_consensus.png', width=2000, height=2000, scale=1)
fig.show()

### Per-sample 3D bar plots

One figure per sample. Bars flagged as `low_expected` (too few co-occurring
cells to trust the ratio) are set to zero. A single global color scale and
axis range are used across samples for comparability.

In [ ]:
# Per-sample 3D neighborhood-enrichment bar plots.
df_pairs_all = pd.read_csv(csvdir + 'ext_fig2_distances.csv')
cats = [c for c in celltype_order if c in df_pairs_all['source'].unique()]
n = len(cats)

# Symmetric global colour and axis range so per-sample plots are comparable.
abs_max = float(np.nanmax(np.abs(df_pairs_all['log2_oe']))) if len(df_pairs_all) else 1.0
norm = mcolors.TwoSlopeNorm(vmin=-abs_max, vcenter=0, vmax=abs_max)
z_range = [0, float(np.ceil(abs_max * 2) / 2)]

_elev, _azim, _r = 40, -215, 4
_eye = dict(x=_r*math.cos(math.radians(_elev))*math.cos(math.radians(_azim)),
            y=_r*math.cos(math.radians(_elev))*math.sin(math.radians(_azim)),
            z=_r*math.sin(math.radians(_elev)))
_font = dict(family='Helvetica', size=5, color='black')
axis_style = dict(showbackground=False, showgrid=False, zeroline=False,
                  showline=True, linecolor='black', tickcolor='black', tickfont=_font)

highlight_pairs = [
    ('B','T'), ('B','B'), ('T','T'), ('T','B'),
    ('B','Neuroblast'), ('Neuroblast','B'), ('T','Neuroblast'), ('Neuroblast','T'),
]

for sample in sorted(df_pairs_all['sample'].unique()):
    df_s = df_pairs_all[df_pairs_all['sample'] == sample]
    mat_log2 = (df_s.pivot(index='source', columns='neighbor', values='log2_oe')
                  .reindex(cats).reindex(cats, axis=1).fillna(0).values)
    # Mask cell-type pairs flagged as low_expected.
    mat_lowex = (df_s.pivot(index='source', columns='neighbor', values='low_expected')
                   .reindex(cats).reindex(cats, axis=1).fillna(True).values)
    mat_log2 = np.where(mat_lowex, 0, mat_log2)

    xpos, ypos = np.meshgrid(np.arange(n), np.arange(n), indexing='ij')
    xpos = xpos.flatten(); ypos = ypos.flatten()
    dz = mat_log2.flatten()

    fig = go.Figure()
    for i in range(len(dz)):
        if dz[i] == 0 or np.isnan(dz[i]):
            continue
        xi, yi = int(xpos[i]), int(ypos[i])
        val = float(dz[i]); h = float(np.abs(val))
        add_full_bar(fig, xi, yi, h, val_to_color(val, norm))
        add_bar_edges(fig, xi, yi, h)
    for src, nbr in highlight_pairs:
        if src in cats and nbr in cats:
            xi, yi = cats.index(src), cats.index(nbr)
            h = float(np.abs(dz[xi * n + yi]))
            if h > 0:
                add_highlight_ring(fig, xi, yi, h, color='white', radius=0.2, width=6)

    fig.update_layout(
        title=f'{sample} 3D heatmap (red = enriched, blue = depleted)',
        scene=dict(aspectmode='manual', aspectratio=dict(x=1.2, y=1.2, z=1.0),
                   xaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats),
                   yaxis=dict(**axis_style, tickvals=list(range(n)), ticktext=cats),
                   zaxis=dict(**axis_style, range=z_range,
                              title=dict(text='|log2(O/E)|', font=_font)),
                   camera=dict(eye=_eye), bgcolor='white'),
        paper_bgcolor='white', plot_bgcolor='white', width=800, height=950,
    )
    fig.write_image(figdir + f'nhood_enrichment_3d_{sample}.png', width=2000, height=2000, scale=1)
    fig.show()